In [ ]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
URL = "https://raw.githubusercontent.com/strotgen/mexico-leaflet/master/states.geojson"
crudo = gpd.read_file(URL)
print(crudo.shape, crudo.crs)      # (32, 4) EPSG:4326
print(crudo.dtypes)

In [ ]:
estados = crudo.rename(columns={"state_code": "cve_ent", "state_name": "nombre"})
esatdos = estados[["cve_ent", "nombre", "geometry"]]
estados["cve_ent"] = estados["cve_ent"].astype(int).astype(str).str.zfill(2)
estados = estados.sort_values("cve_ent").reset_index(drop=True)
print(estados.head(3))

In [ ]:
anios = range(2017, 2026)
rng = np.random.default_rng(42)

#proyectamos la latitud del centro para que quede bien
lat = estados.to_crs(6372).centroid.to_crs(4326).y.to_numpy()

#una base inventada para que haya mas valores en el sur
base= 40 - 2.0 * (lat-lat.mean()) + rng.normal(0, 4, len(lat))

filas = []

for i, cve in enumerate(estados["cve_ent"]): 
    hab = rng.integers(700_000, 17_000_000)
    for anio in anios: 
        filas.append({"cve_ent": cve, "anio": anio, 
                      "habitantes": int(hab * (1 + 0.01 * (anio - 2017))),
                      "casos": int(rng.poisson(max(base[i], 1) * hab / 100_000))})

panel = pd.DataFrame(filas)
panel["tasa"] = (panel["casos"] / panel["habitantes"] * 100_000).round(2)
print(panel.shape)
print(panel.head(3))

In [ ]:
#investigar sobre como funcionan los crs
estados["geometry"] = estados.make_valid(method="structure", keep_collapsed=False)
# print("valias", estados.is_valid())
print(estados.is_valid.sum())

In [ ]:
estados["km2"] = (estados.to_crs(6372).area / 1e6).round(0)
print(estados.sort_values("km2")[["nombre", "km2"]].head(3))

In [ ]:
mapa_2024 = estados.merge(panel[panel["anio"] == 2024], on="cve_ent", how="left", validate="one_to_one")

print(type(mapa_2024).__name__, mapa_2024.shape, "faltantes: ", mapa_2024["tasa"])

In [ ]:
print(mapa_2024)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
mapa_2024.plot(column="tasa", cmap="YlOrRd", legend=True,
               edgecolor="white", linewidth=0.4, ax=axes[0])
axes[0].set_title("Escala continua")
mapa_2024.plot(column="tasa", cmap="YlOrRd", scheme="Quantiles", k=4, 
               legend=True, edgecolor="white", linewidth=0.4, ax=axes[1], 
               legend_kwds={"loc": "lower left", "fmt": "{:.1f}", 
                            "title": "Tasa(inventada)"})
axes[1].set_title("Cuartiles")
for ax in axes: 
    ax.set_axis_off()
fig.show()

            